# Phase 5 — Calibration and Stability

Champion: **GBM application-only**. LendingClub's `grade`/`sub_grade`/`int_rate` are that
platform's own risk output, so an originator scoring its own applicants has no counterpart
to them. GBM full is kept as a benchmark that measures what those columns add, not as the
model taken forward.

Discrimination is settled (OOT AUC 0.6972, Gini 0.3945). Nothing so far has checked whether
a predicted PD of 0.12 is followed by 12% defaults - which is what every downstream decision
depends on.


In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import yaml

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.features.build_dataset import (
    APPLICATION_FEATURES, application_features, assemble_feature_matrix,
)
from credit_risk.models.gbm import predict_gbm, train_gbm
from credit_risk.models.scorecard import predict_scorecard, train_scorecard
from credit_risk.evaluation.calibration import (
    Calibrator, brier_decomposition, central_tendency_shift,
    expected_calibration_error, pd_to_score, reliability_table,
)
from credit_risk.evaluation.stability import population_stability_index, psi_report

pl.Config.set_tbl_rows(40)

CONFIG_PATH = Path("../configs/base.yaml")
PARAMS_PATH = Path("../configs/gbm_best_params_application.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")


In [2]:
final = assemble_feature_matrix(
    build_target(load_raw_accepted_loans(DATA_PATH), CONFIG_PATH), CONFIG_PATH
)
splits = {name: final.filter(pl.col("split") == name) for name in ("train", "validation", "oot_test")}

features = application_features(final)
params = yaml.safe_load(open(PARAMS_PATH))
model, features = train_gbm(splits["train"], splits["validation"], params=params, features=features)

pred = {name: predict_gbm(model, features, df) for name, df in splits.items()}
y = {name: df["default_flag"].to_numpy() for name, df in splits.items()}

for name in splits:
    print(name, "mean_pd", round(float(pred[name].mean()), 4), "actual", round(float(y[name].mean()), 4))


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[659]	valid_0's auc: 0.706944
train mean_pd 0.0892 actual 0.0892
validation mean_pd 0.0939 actual 0.107
oot_test mean_pd 0.0892 actual 0.1136


## 1. Is the champion calibrated out of the box?

`mean_pd` vs `actual` above is the crude check. Train should match almost exactly - the model
fitted it. The gap on validation and OOT is the real question: the model learned an 8.9% base
rate and is being applied to populations running at 10.7% and 11.4%.

`gap` below is predicted minus observed, so positive means the model is too pessimistic.


In [3]:
for name in ("validation", "oot_test"):
    print(f"--- {name} ---")
    print(reliability_table(y[name], pred[name], n_bins=10))
    print("ECE", round(expected_calibration_error(y[name], pred[name]), 4))
    print(brier_decomposition(y[name], pred[name]), "\n")


--- validation ---
shape: (10, 5)
┌─────┬───────┬────────────────┬───────────────┬───────────┐
│ bin ┆ n     ┆ mean_predicted ┆ observed_rate ┆ gap       │
│ --- ┆ ---   ┆ ---            ┆ ---           ┆ ---       │
│ i64 ┆ i64   ┆ f64            ┆ f64           ┆ f64       │
╞═════╪═══════╪════════════════╪═══════════════╪═══════════╡
│ 0   ┆ 42110 ┆ 0.018865       ┆ 0.019045      ┆ -0.00018  │
│ 1   ┆ 42110 ┆ 0.033171       ┆ 0.035716      ┆ -0.002545 │
│ 2   ┆ 42110 ┆ 0.044911       ┆ 0.050487      ┆ -0.005576 │
│ 3   ┆ 42110 ┆ 0.056851       ┆ 0.067015      ┆ -0.010164 │
│ 4   ┆ 42110 ┆ 0.069881       ┆ 0.082379      ┆ -0.012499 │
│ 5   ┆ 42109 ┆ 0.084791       ┆ 0.10233       ┆ -0.017538 │
│ 6   ┆ 42109 ┆ 0.102774       ┆ 0.118834      ┆ -0.016061 │
│ 7   ┆ 42109 ┆ 0.126247       ┆ 0.146073      ┆ -0.019826 │
│ 8   ┆ 42109 ┆ 0.160649       ┆ 0.183856      ┆ -0.023208 │
│ 9   ┆ 42109 ┆ 0.24079        ┆ 0.264124      ┆ -0.023334 │
└─────┴───────┴────────────────┴───────────────┴───

## 2. Calibrate on validation, measure on OOT

The calibrator is fitted on 2015 and applied to 2016. Fitting it on train would re-learn the
fit the model already has and report near-perfect calibration that does not exist.

Isotonic and Platt are both fitted so the choice is evidence-based: isotonic corrects any
monotone distortion but cannot extrapolate past its fitted range; Platt only shifts and
rescales, which is more stable in thin tails.


In [4]:
results = {}
for method in ("platt", "isotonic"):
    calibrator = Calibrator(method).fit(y["validation"], pred["validation"])
    calibrated = calibrator.transform(pred["oot_test"])
    results[method] = calibrated
    print(method,
          "ECE", round(expected_calibration_error(y["oot_test"], calibrated), 4),
          "mean_pd", round(float(calibrated.mean()), 4),
          "min_pd", f"{calibrated.min():.2e}",
          "distinct", len(np.unique(calibrated)))

print("uncalibrated ECE", round(expected_calibration_error(y['oot_test'], pred['oot_test']), 4))
print("actual OOT rate ", round(float(y['oot_test'].mean()), 4))


platt ECE 0.0118 mean_pd 0.1017 min_pd 3.56e-03 distinct 434407
isotonic ECE 0.0117 mean_pd 0.1018 min_pd 1.00e-04 distinct 212
uncalibrated ECE 0.0244
actual OOT rate  0.1136


In [5]:
# Platt is preferred unless isotonic beats it by more than noise: it preserves ranking
# exactly, extrapolates, and cannot collapse a block of loans onto a single PD.
from credit_risk.evaluation.metrics import auc

calibrated = results["platt"]
for method, values in results.items():
    print(method, "AUC", round(auc(y["oot_test"], values), 4),
          "| ECE", round(expected_calibration_error(y["oot_test"], values), 4))
print("uncalibrated AUC", round(auc(y["oot_test"], pred["oot_test"]), 4))


platt AUC 0.6972 | ECE 0.0118
isotonic AUC 0.697 | ECE 0.0117
uncalibrated AUC 0.6972


## 3. Calibration per term — the horizon bias

H=24 truncates the two terms unequally: it captures roughly 60% of eventual 36-month defaults
but only 42% of 60-month ones. A single calibration applied to both therefore understates
60-month risk by more than it understates 36-month risk.

If `gap` differs materially between terms, calibrate per term rather than pooled.


In [6]:
oot = splits["oot_test"].with_columns(pl.Series("pd_calibrated", calibrated))

for term in (36, 60):
    part = oot.filter(pl.col("term_months") == term)
    yt = part["default_flag"].to_numpy()
    pt = part["pd_calibrated"].to_numpy()
    print(f"term={term}  n={part.height}  mean_pd={pt.mean():.4f}  observed={yt.mean():.4f} "
          f"gap={pt.mean() - yt.mean():+.4f}  ECE={expected_calibration_error(yt, pt):.4f}")


term=36  n=323495  mean_pd=0.0927  observed=0.1039 gap=-0.0112  ECE=0.0112
term=60  n=110912  mean_pd=0.1280  observed=0.1417 gap=-0.0136  ECE=0.0136


In [7]:
# Per-term calibrators, fitted on validation within each term.
valid = splits["validation"]
for term in (36, 60):
    v = valid.filter(pl.col("term_months") == term)
    o = oot.filter(pl.col("term_months") == term)
    idx_v = valid["term_months"].to_numpy() == term
    idx_o = oot["term_months"].to_numpy() == term
    cal = Calibrator("platt").fit(v["default_flag"].to_numpy(), pred["validation"][idx_v])
    pt = cal.transform(pred["oot_test"][idx_o])
    yt = o["default_flag"].to_numpy()
    print(f"term={term}  mean_pd={pt.mean():.4f}  observed={yt.mean():.4f} "
          f"gap={pt.mean() - yt.mean():+.4f}  ECE={expected_calibration_error(yt, pt):.4f}")


term=36  mean_pd=0.0923  observed=0.1039 gap=-0.0116  ECE=0.0116
term=60  mean_pd=0.1288  observed=0.1417 gap=-0.0129  ECE=0.0129


## 4. Central tendency and point scale

Calibrating to 2016 is fitting to one vintage. A production PD is usually anchored to a
long-run average instead, which is a pure intercept shift in log-odds: ranking is preserved,
only the level moves. The long-run rate here is the mean across the three observed vintages -
a placeholder, since three vintages is not a cycle.

`pd_to_score` then converts PD to points. A PD without a point scale is not yet a scorecard:
cutoffs, overrides and production monitoring are all expressed in points.


In [8]:
long_run_rate = float(np.mean([y[name].mean() for name in splits]))
shift = central_tendency_shift(calibrated, long_run_rate)
logits = np.log(np.clip(calibrated, 1e-9, 1 - 1e-9) / (1 - np.clip(calibrated, 1e-9, 1 - 1e-9)))
anchored = 1 / (1 + np.exp(-(logits + shift)))

print(f"long-run rate {long_run_rate:.4f}  log-odds shift {shift:+.4f}")
print(f"mean PD  {calibrated.mean():.4f} -> {anchored.mean():.4f}")
print("AUC unchanged:", round(auc(y["oot_test"], anchored), 4))

scores = pd_to_score(anchored, pdo=20, base_score=600, base_odds=50.0)
print("score range", round(float(scores.min())), "-", round(float(scores.max())),
      " median", round(float(np.median(scores))))


long-run rate 0.1033  log-odds shift +0.0176
mean PD  0.1017 -> 0.1033
AUC unchanged: 0.6972
score range 473 - 649  median 555


In [9]:
# Bad rate by score band - the table a credit committee actually reads.
band = pl.DataFrame({"score": scores, "default_flag": y["oot_test"]}).with_columns(
    (pl.col("score") / 20).floor().cast(pl.Int32).alias("band")
)
band.group_by("band").agg(
    pl.len().alias("n"),
    pl.col("score").min().round(0).alias("score_from"),
    pl.col("default_flag").mean().round(4).alias("bad_rate"),
).sort("band", descending=True)


band,n,score_from,bad_rate
i32,u32,f64,f64
32,32,640.0,0.0
31,2101,620.0,0.0081
30,13973,600.0,0.0137
29,50522,580.0,0.0324
28,115923,560.0,0.064
27,143571,540.0,0.1163
26,86201,520.0,0.1927
25,20644,500.0,0.2999
24,1424,481.0,0.3869


## 5. Stability (PSI)

Score PSI answers whether the population the model sees in 2016 still resembles 2013-2014.
Feature PSI localises any shift. Bands: <0.10 stable, 0.10-0.25 watch, >0.25 material.

Read these against the discrimination result, not on their own: OOT AUC held at 0.6972, so a
material PSI here would mean the population moved without the score's ranking breaking - a
reason to re-calibrate, not to retrain.


In [10]:
score_psi = population_stability_index(
    pl.Series(pred["train"]), pl.Series(pred["oot_test"]), n_bins=10
)
print("score PSI train -> oot_test:", round(score_psi, 4))
print("score PSI train -> validation:",
      round(population_stability_index(pl.Series(pred["train"]), pl.Series(pred["validation"])), 4))


score PSI train -> oot_test: 0.0004
score PSI train -> validation: 0.0061


In [11]:
report = psi_report(splits["train"], splits["oot_test"], APPLICATION_FEATURES)
print(report.to_pandas().to_string(index=False))
report.write_csv("../docs/psi_application_features.csv")


              feature      psi   band
     percent_bc_gt_75 0.081146 stable
       bc_open_to_buy 0.045292 stable
 acc_open_past_24mths 0.031344 stable
 mo_sin_old_rev_tl_op 0.027321 stable
                  dti 0.021312 stable
      tot_hi_cred_lim 0.010912 stable
           annual_inc 0.010137 stable
 mths_since_recent_bc 0.008864 stable
       fico_range_low 0.007475 stable
          term_months 0.006092 stable
       mo_sin_rcnt_tl 0.005529 stable
mths_since_recent_inq 0.003271 stable
